# Computational Workflows for biomedical data

Welcome to the course Computational Workflows for Biomedical Data. Over the next two weeks, you will learn how to leverage nf-core pipelines to analyze biomedical data and gain hands-on experience in creating your own pipelines, with a strong emphasis on Nextflow and nf-core.

Course Structure:

- Week 1: You will use a variety of nf-core pipelines to analyze a publicly available biomedical study.
- Week 2: We will shift focus to learning the basics of Nextflow, enabling you to design and implement your own computational workflows.<br>
- Final Project: The last couple of days, you will apply your knowledge to create a custom pipeline for analyzing biomedical data using Nextflow and the nf-core template.

## Basics

If you have not installed all required software, please do so now asap!


If you already installed all software, please go on and start answering the questions in this notebook. If you have any questions, don't hesitate to approach us.

1. What is nf-core?

nf-core is a project across many groups, which provides a set of bioinformatics pipelines using Netxflow.

2. How many pipelines are there currently in nf-core?

157 (101 released, 43 under developement, 13 archived)

3. Are there any non-bioinformatic pipelines in nf-core?

Yes, for example troughgraph, used to analyze permafrost.

4. Let's go back a couple of steps. What is a pipeline and what do we use it for?

A pipeline is a set of automated steps, processing data from a given input to a desired output. It makes the processing of data more efficient and reproducible, clearly organizes the process and helps with error reduction.

5. Why do you think nf-core adheres to strict guidelines?

The guidelines makes nf-core easier to understand for different user and ensures high quality as well as the possibility to use the pipelines in different environments.

6. What are the main features of nf-core pipelines?

nf-core pipelines provide extensive documentation and stable releases. They are open source and their quality is ensured through continuous integration testing. They are also portabl, can run in different computing environments and they come as packaged software.

## Let's start using the pipelines

1. Find the nf-core pipeline used to measure differential abundance of genes

https://nf-co.re/differentialabundance/2.0.0/

In [ ]:
# run the pipeline in a cell 
# to run bash in jupyter notebooks, simply use ! before the command
# e.g.

!pwd


# For the tasks in the first week, please use the command line to run your commands and simply paste the commands you used in the respective cells!


In [ ]:
# run the pipeline in the test profile using docker containers
# make sure to specify the version you want to use (use the latest one)


!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker -c local.config --outdir results


# runtime: 10m 19.5s

In [ ]:
# repeat the run. What did change?

!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker -c local.config --outdir results

# runtime: 8m 24.1s

What changed?

 The second run was slightly faster (~8 min vs. ~10 min), most likely because the Docker images and the pipeline code had already been downloaded during the first run, which reduced waiting time. The actual computation time of the individual steps should be roughly the same, because  all the steps are executed again.

In [ ]:
# now set -resume to the command. What did change?

!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker -c local.config --outdir results -resume

# runtime: 35.1s

Check out the current directory. Next to the outdir you specified, what else has changed?

Besides the output directory, a work/ directory was created, which contains a separate subfolder for each task with its intermediate files, scripts and logs. It also created a hidden .nextflow/ folder that stores the cache and run history (used by -resume), as well as .nextflow.log files that log each run.

In [ ]:
# delete the work directory and run the pipeline again using -resume. What did change?

!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker -c local.config --outdir results -resume

# 

What changed?

After deleting the work directory -resume could not reuse any previous results, because the cached task outputs are stored in work/. Nextflow still had the run history in .nextflow/, but because the corresponding files were missing, all processes had to be executed again. Therefore the runtime (~9 min) was similar to a normal run without -resume and much longer than the previous -resume run, where all tasks were taken from the cache.

## Lets look at the results

### What is differential abundance analysis?

Differential abundance analysis is used to find genes (or other features like proteins) that have a significantly different abundance between conditions, for example treated vs control samples. For RNA-seq the input is a count matrix with the reads per gene for each sample, which gets normalized first because of differences like sequencing depth. Then a statistical model like DESeq2 or limma tests for every gene if the difference between the groups is bigger than what you would expect by chance. The result is usually a log2 fold change and an adjusted p-value for each gene, so you can see which genes are up- or downregulated and how reliable this is. In our test run the pipeline used DESeq2 to compare the treatment groups of the mouse dataset SRP254919.

Give the most important plots from the report:

*PCA plot*

![PCA plot](important_plots_examples/pca2d.png) -> Shows the samples on the first two principal components, colored by treatment group, to see how similar they are.

*2. Sample clustering dendrogram*

![Sample clustering dendrogram](important_plots_examples/sample_dendrogram.png) -> Hierarchical clustering of the samples based on the 500 most variable genes; samples joined lower are more similar.


*3. Volcano plot*

![Volcano plot](important_plots_examples/volcano.png) -> Shows log2 fold change vs. significance (-log10 padj) for each gene; genes passing both thresholds are highlighted.

In [ ]:
compare two plots
plts: 2d pca  and clustering heatmap

In [ ]:
# Comparison: PCA plot vs. volcano plot

The PCA shows that the treatment groups only separate on PC2 (22.1%), while the largest variation (PC1, 45.3%) comes from differences between the sample pairs. This agrees with the volcano plot; most genes have a log2 fold change close to 0 and are not significant, and only a few genes pass both the fold change and the padj = 0.05 threshold. So both plots suggest that there is a treatment effect, but it is relatively small and affects only a small number of genes. One reason could also be that the test dataset only contains a subset of genes (top 1000).